### This is a trial to run the meta-learning agent in the trial environment
Population weights: Uniform

In [1]:
import os, sys
os.environ["PYTHONPATH"] = "/home/rifnyrachman7"
sys.path.append("/home/rifnyrachman7/_merlion/")
sys.path.append("/home/rifnyrachman7/_merlion/ds-project-messiah/src")
sys.path.append("/home/rifnyrachman7/")

In [2]:
from gymnasium.wrappers import TimeLimit
import gymnasium as gym
from pathlib import Path

# Import both modules
from _metamorl.algo_ray.rllib.algorithms.maml.merlion_main_multitasks import MERLION, MERLIONConfig
from _metamorl.algo_ray.rllib.algorithms.maml.callback_save_population import _find_archive , MERLIONWithSaveCallbacks

 #v2.3.1 #use algo from local repo
import ray
from ray import air, tune
from ray.tune.registry import register_env
from gymnasium.wrappers import TimeLimit
from ray.air import Checkpoint
from ray.rllib.algorithms.algorithm import Algorithm

# Import the required modules

from messiah.environment import Environment
from messiah.agents.processing import MaxAvailableProcessingAgent

import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt

import tempfile
import pickle
import json


#from utils.plot_utils import PlotUtils

In [3]:
#import generated environment
from _merlion.random_env import moderate_env_merlion
moderate_sc = moderate_env_merlion.ModerateSC()
#simple_sc.save_filename = "/home/rifnyrachman7/_merlion/random_env/simple_reward.pkl"

In [4]:
%load_ext autoreload
%autoreload 2

In [5]:
def save_full_checkpoints_for_population(algo, out_dir: str):
    archive = _find_archive(algo)
    if archive is None:
        print("[SavePopulation] MERLION archive not found on algorithm; skipping.")
        return

    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    pol = algo.get_policy()
    P = int(getattr(archive, "population_size", len(getattr(archive, "archive", []))))

    for i in range(P):
        item = archive.get_archive_item(i)
        if item is None:
            continue

        theta_i = item["theta"]
        w_i = np.asarray(item["weight"], dtype=np.float32)   # <- correct key

        # save the full RLlib checkpoint for policy i
        pol.set_weights(theta_i)
        ckpt_dir = out / f"checkpoint_policy_{i}"
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        path = algo.save(str(ckpt_dir))
        print(f"[SavePopulation] Saved RLlib checkpoint for i={i} → {path}")

        # save the associated weight next to it
        with open(ckpt_dir / "weight.json", "w") as f:
            json.dump([float(x) for x in w_i.ravel().tolist()], f)

In [6]:
ray.shutdown()
ray.init()

register_env("moderate_sc", lambda cfg: moderate_env_merlion.make_env(cfg))

rollout_fragment_length = 32 #default 32

2026-01-28 10:20:10,132	WARNING utils.py:640 -- Detecting docker specified CPUs. In previous versions of Ray, CPU detection in containers was incorrect. Please ensure that Ray has enough CPUs allocated. As a temporary workaround to revert to the prior behavior, set `RAY_USE_MULTIPROCESSING_CPU_COUNT=1` as an env var before starting Ray. Set the env var: `RAY_DISABLE_DOCKER_CPU_WARNING=1` to mute this warning.
2026-01-28 10:20:10,133	WARNING utils.py:652 -- Ray currently does not support initializing Raywith fractional cpus. Your num_cpus will be truncated from 13.68 to 13.
2026-01-28 10:20:10,134	WARNING services.py:1780 -- WARNING: The object store is using /tmp instead of /dev/shm because /dev/shm has only 67108864 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=10.24gb' to 'docker run' (or add it to the run_options list in a Ray clu

### Adaptation and Metalearning Phase

In [7]:
seed_base = 99

config = (
    MERLIONConfig()
    .rollouts(
        num_rollout_workers=4, rollout_fragment_length=rollout_fragment_length
    )
    .framework("torch")
    .environment("moderate_sc", clip_actions=True)
    .training(
        inner_adaptation_steps=4,
        meta_batch_size=4, # follow rollout_workers in MIRACL that represent task batch number
        maml_optimizer_steps=10,
        gamma=0.99,
        lambda_=1.0,
        lr=0.001,
        vf_loss_coeff=0.5,
        inner_lr=0.03,
        use_meta_env=True,
        clip_param=0.3,
        kl_target=0.01,
        kl_coeff=0.001,
        model=dict(fcnet_hiddens=[64, 64]),
        train_batch_size=rollout_fragment_length,
        
        #MERLION specific parameters
        population_size = 21, #default # TO DO: ablation for this
        num_objectives = 3,
    )
    .evaluation(
        evaluation_num_workers=1,
        evaluation_interval=100,
        enable_async_evaluation=True,
    )
    .callbacks(MERLIONWithSaveCallbacks)
)

In [8]:
local_dir = "/home/rifnyrachman7/_merlion/training_data/moderate_sc"
num_iterations = 22 #default 500

results = {}
runs = 10 #run multiple times for model robustness
for k in range(runs):
    print(f"RUNNING NUMBER-{k}")
    
    tuner = tune.Tuner(
        MERLION,
        param_space=config.to_dict(),
        run_config=air.RunConfig(
            name=f"merlion_moderate_{k}",
            local_dir = local_dir,
            stop={"training_iteration": num_iterations},
            failure_config=air.FailureConfig(fail_fast="raise"),
            log_to_file=True,
            checkpoint_config=air.CheckpointConfig(checkpoint_at_end=True, num_to_keep=1),  # ensure a checkpoint exists
        )
    )
    results[k] = tuner.fit()
    
    # #save the best checkpoint for each run
    best = results[k].get_best_result()
    algo = Algorithm.from_checkpoint(best.checkpoint)
    
    out_dir = os.path.join(local_dir, f"metapolicies-{k}")
    os.makedirs(out_dir, exist_ok=True)
    save_full_checkpoints_for_population(algo, out_dir)

RUNNING NUMBER-0


/home/rifnyrachman7/.local/lib/python3.10/site-packages/ray/tune/execution/trial_runner.py:402: UserWarning: fail_fast='raise' detected. Be careful when using this mode as resources (such as Ray processes, file descriptors, and temporary files) may not be cleaned up properly. To use a safer mode, use fail_fast=True.
  warnings.warn(
(MERLION pid=33699) 2026-01-21 23:20:00,119	WARNING algorithm_config.py:596 -- Cannot create MERLIONConfig from given `config_dict`! Property __stdout_file__ not supported.
(MERLION pid=33699) 2026-01-21 23:20:00,119	WARNING algorithm_config.py:596 -- Cannot create MERLIONConfig from given `config_dict`! Property __stdout_file__ not supported.
(MERLION pid=33699) 2026-01-21 23:20:00,629	INFO algorithm.py:506 -- Current log_level is WARN. For more information, set 'log_level': 'INFO' / 'DEBUG' or use the -v and -vv flags.
(MERLION pid=33699) 2026-01-21 23:20:00,629	INFO algorithm.py:506 -- Current log_level is WARN. For more information, set 'log_level': 'IN

(MERLION pid=33699) 
(MERLION pid=33699) === POPULATION OBJECTIVE BREAKDOWN (Iter 0) ===
(MERLION pid=33699) 00 w=[0.052,0.460,0.489] obj=[   0.23,   0.62,   0.70] sc=    0.640  <- OK
(MERLION pid=33699) 01 w=[0.180,0.495,0.325] obj=[   0.22,   0.60,   0.72] sc=    0.569  <- OK
(MERLION pid=33699) 02 w=[0.343,0.085,0.573] obj=[   0.31,   0.58,   0.72] sc=    0.568  <- OK
(MERLION pid=33699) 03 w=[0.332,0.462,0.206] obj=[   0.08,   0.61,   0.68] sc=    0.451  <- OK
(MERLION pid=33699) 04 w=[0.386,0.365,0.249] obj=[   0.25,   0.60,   0.72] sc=    0.496  <- OK
(MERLION pid=33699) 05 w=[0.044,0.632,0.324] obj=[   0.10,   0.62,   0.73] sc=    0.634  <- OK
(MERLION pid=33699) 06 w=[0.182,0.728,0.090] obj=[   0.36,   0.63,   0.72] sc=    0.586  <- OK
(MERLION pid=33699) 07 w=[0.745,0.206,0.049] obj=[   0.16,   0.64,   0.75] sc=    0.285  <- OK
(MERLION pid=33699) 08 w=[0.503,0.122,0.375] obj=[   0.28,   0.63,   0.70] sc=    0.480  <- OK
(MERLION pid=33699) 09 w=[0.189,0.087,0.724] obj=[   0.2

Trial name,adaptation_delta,agent_timesteps_total,connector_metrics,custom_metrics,date,done,episode_len_mean,episode_media,episode_reward_max,episode_reward_max_adapt_1,episode_reward_max_adapt_2,episode_reward_max_adapt_3,episode_reward_max_adapt_4,episode_reward_mean,episode_reward_mean_adapt_1,episode_reward_mean_adapt_2,episode_reward_mean_adapt_3,episode_reward_mean_adapt_4,episode_reward_min,episode_reward_min_adapt_1,episode_reward_min_adapt_2,episode_reward_min_adapt_3,episode_reward_min_adapt_4,episodes_this_iter,episodes_total,experiment_id,hostname,info,iterations_since_restore,node_ip,num_faulty_episodes,num_healthy_workers,perf,pid,policy_reward_max,policy_reward_mean,policy_reward_min,sampler_perf,time_since_restore,time_this_iter_s,time_total_s,timers,timestamp,timesteps_since_restore,timesteps_this_iter,timesteps_total,training_iteration,trial_id,warmup_time
MERLION_moderate_sc_b3133_00000,0.417812,0,"{'ObsPreprocessorConnector_ms': 0.004298562635910484, 'StateBufferConnector_ms': 0.0032357347795482125, 'ViewRequirementAgentConnector_ms': 0.0790496532068231}",{},2026-01-22_00-02-21,False,100,{},55.4744,59.4041,57.9112,56.901,52.8523,50.5232,47.4746,51.4409,48.7651,50.941,42.0055,36.474,46.7133,43.7874,46.3781,441,7056,80186df90bdd4f7e8571688ee284db1b,workspace-580f8f1b-0605-48d2-b4f5-8608dc1d24e9-0,"{'learner': {'default_policy': {'learner_stats': {'allreduce_latency': 0.0, 'grad_gnorm': 1.0, 'worker_loss': 43.91499328613281}, 'model': {}, 'custom_metrics': {}, 'num_agent_steps_trained': 2000, 'num_grad_updates_lifetime': 864, 'diff_num_grad_updates_vs_sampler_policy': 9.0}}, 'num_steps_sampled': 691500, 'num_steps_trained_this_iter': 46100, 'num_steps_trained': 691500}",16,10.131.33.49,0,5,"{'cpu_util_percent': 15.64304932735426, 'ram_util_percent': 26.494618834080715}",33699,{},{},{},"{'mean_raw_obs_processing_ms': 0.5635955228242222, 'mean_inference_ms': 0.6940094888668241, 'mean_action_processing_ms': 0.13136642395582934, 'mean_env_wait_ms': 1.8608043480442586, 'mean_env_render_ms': 0.0}",2528.88,156.76,2528.88,{},1769040141,737600,46100,737600,16,b3133_00000,11.3792


(MERLION pid=33699) [MERLION] timesteps_this_iter=46100, timesteps_total=46100
(MERLION pid=33699) 
(MERLION pid=33699) === POPULATION OBJECTIVE BREAKDOWN (Iter 1) ===
(MERLION pid=33699) 00 w=[0.838,0.151,0.011] obj=[   0.33,   0.59,   0.71] sc=    0.378  <- OK
(MERLION pid=33699) 01 w=[0.171,0.555,0.274] obj=[   0.27,   0.59,   0.70] sc=    0.569  <- OK
(MERLION pid=33699) 02 w=[0.919,0.081,0.000] obj=[   0.33,   0.60,   0.72] sc=    0.348  <- OK
(MERLION pid=33699) 03 w=[0.440,0.192,0.368] obj=[   0.21,   0.61,   0.71] sc=    0.470  <- OK
(MERLION pid=33699) 04 w=[0.369,0.292,0.339] obj=[   0.27,   0.63,   0.73] sc=    0.531  <- OK
(MERLION pid=33699) 05 w=[0.420,0.189,0.391] obj=[   0.16,   0.62,   0.70] sc=    0.459  <- OK
(MERLION pid=33699) 06 w=[0.604,0.224,0.172] obj=[   0.24,   0.61,   0.71] sc=    0.406  <- OK
(MERLION pid=33699) 07 w=[0.372,0.394,0.233] obj=[   0.11,   0.61,   0.71] sc=    0.447  <- OK
(MERLION pid=33699) 08 w=[0.153,0.000,0.847] obj=[   0.38,   0.58,   0.7

2026-01-22 00:04:31,247	WARNING tune.py:146 -- Stop signal received (e.g. via SIGINT/Ctrl+C), ending Ray Tune run. This will try to checkpoint the experiment state one last time. Press CTRL+C (or send SIGINT/SIGKILL/SIGTERM) to skip. 
2026-01-22 00:04:31,254	ERROR tune.py:794 -- Trials did not complete: [MERLION_moderate_sc_b3133_00000]
2026-01-22 00:04:31,254	INFO tune.py:798 -- Total run time: 2674.36 seconds (2674.35 seconds for the tuning loop).
2026-01-22 00:04:31,255	WARNING tune.py:804 -- Experiment has been interrupted, but the most recent state was saved. You can continue running this experiment by passing `resume=True` to `tune.run()`

KeyboardInterrupt



#### TO TRY NEXT:
* Narrowing the profit range for normalisation >> hopefully when the the agent sees the profit has bigger portion, it affects more to the final rewards, then the agent will consider to put more weights into profit!

#### NOTE, My observation on MERLION behaviour:
It tends to increase the reward on the reward vector value that contributes more in the final scalar reward, which is expected. Consequently, it is sensitive to normalisation range. If we set normalisation range too broad --> initial reard contribution in the beginning will be low --> Merlion thinks that this reward has less significant impact to final scalar reward --> allocate the weights to other reward vector values and decreases the weight for this one.

### METAMORL
timesteps = W (S + 1) H

### MERLION
H = max(timestep/episode, rollout_fragment_length)
S = inner_adaptation_steps
W = num_rollout_workers
P = population_size
J = meta_batch_size

timesteps = W (S + 1) H + P (J (S + 1) H + H)
            {metaMORL-like} {population-stage}

A timestep represents an agent-environment interaction sampling.

In MERLION, the task batch size is defined in meta_batch_size, while num_rollout_workers only determine the parallel computation for speed. In Metamorl, the n_tasks is defined based on num_rollout_worker, so it is attached to the core number used.

In MERLION, the iteration is calculated for the overall population, hence each individual solution learns much less iteration than Metamorl. The justification is because each individual solution learns locally, focusing within narrower search space.

### How the iteration works:

each MERLION individual θᵢ gets fewer total updates than the single MetaMORL meta-policy,

“each individual solution learns locally, focusing within a narrower search space”

because the sample budget is spread across many individuals rather than concentrated on one.

So:

As a theoretical / structural statement: say

“Per global iteration, MERLION trains all individuals; cost scales with P×meta_batch_size.”

As a fair-sample comparison statement: say

“For a fixed environment step budget, each MERLION individual effectively sees fewer updates than a single MetaMORL meta-policy, trading depth per policy for coverage/diversity across the population.”

PS: Total running time in MERLION might more than MetaMORL since aside from metapolicy update (inner and outer), it also involves weight update thru evolutionary mechanism.

NOTE: Currently, MERLION still contains MetaMORL-like update which is actually algorithmically unnecessary but needed. Hence, these stage can be ignored for iteration and timing.

What happens if you “remove Phase 1”? If by “remove Phase 1” you mean: delete inner_adaptation_steps transform, stop producing (samples, metrics_dict), and leave MERLIONMetaUpdate.__call__ untouched,

then:

data_tuple will no longer be a 2-tuple, _shared_samples = data_tuple[0] will be wrong, you’ll get shape/TypeErrors → so yes, bugs.

This is not because of Ray API, but because your execution_plan and MERLIONMetaUpdate are coupled.

#### For now, let's just do interpolation to ignore MetaMORL update and upgrade version later maybe.

### Try to fine-tune using stable-baselines3

In [8]:
from stable_baselines3 import PPO
from stable_baselines3.ppo.policies import MlpPolicy
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from _merlion.fine_tuning.ft_moderate_env_merlion import TestModerateSC
from _metamorl.algo_ray.rllib.algorithms.maml import merlion_finetuning_sb3_independent as ft

In [9]:
# (Re)start Ray cleanly
ray.shutdown()
ray.init(ignore_reinit_error=True, include_dashboard=False)

def make_env_ray(config):
    """
    RLlib factory: receives a dict `config` and must return a *Gymnasium Env instance*.
    We wrap with TimeLimit here so both RLlib and Gym share the same horizon behavior.
    """
    base = TestModerateSC(**config)            # pass config keys into your env
    max_steps = int(config.get("max_steps", 100))
    return TimeLimit(base, max_episode_steps=max_steps)

register_env("testSC", make_env_ray)
register_env("moderate_sc", lambda cfg: moderate_env_merlion.make_env(cfg)) # register meta-env

2026-01-28 10:20:21,530	WARNING utils.py:640 -- Detecting docker specified CPUs. In previous versions of Ray, CPU detection in containers was incorrect. Please ensure that Ray has enough CPUs allocated. As a temporary workaround to revert to the prior behavior, set `RAY_USE_MULTIPROCESSING_CPU_COUNT=1` as an env var before starting Ray. Set the env var: `RAY_DISABLE_DOCKER_CPU_WARNING=1` to mute this warning.
2026-01-28 10:20:21,531	WARNING utils.py:652 -- Ray currently does not support initializing Raywith fractional cpus. Your num_cpus will be truncated from 13.68 to 13.
2026-01-28 10:20:21,533	WARNING services.py:1780 -- WARNING: The object store is using /tmp instead of /dev/shm because /dev/shm has only 67108864 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=10.24gb' to 'docker run' (or add it to the run_options list in a Ray clu

In [10]:
# Optional: re-register safely if re-running cells
try:
    gym.spec("testSC")  # raises if not registered
    # Gymnasium 0.29.x: remove existing spec to allow re-register
    del gym.envs.registration.registry["testSC"]
except Exception:
    pass

def make_env_gym(**kwargs):
    base = TestModerateSC(**kwargs)
    max_steps = int(kwargs.get("max_steps", 100))
    return TimeLimit(base, max_episode_steps=max_steps)

gym.register(
    id="testSC",
    entry_point=make_env_gym,   # callable that returns an Env
)

Finetuning timesteps = P x (M + 1) x K x W

P: population size

M: number of local perturbation

K: env steps per perturbation

W: rollout workers

In [ ]:
#finetuning timesteps = 10 x 5 x 100 x 1 = 5000 timesteps
runs = 10
for k in range(8, runs):
    start_time = time.time()
    ft.finetune_with_local_perturbations(
        pop_dir=f"/home/rifnyrachman7/_merlion/training_data/moderate_sc/metapolicies-{k}",
        env_id="testSC",
        out_dir=f"/home/rifnyrachman7/_merlion/training_data/moderate_sc/finetuning_sb3-{k}",
        total_steps=16000, # default 5000
        record_every=512, # default 1
        eval_episodes=50, # default 50
        m_perturb=2,
        eps=0.05,
        seed=7,
        skip_existing=True,
    )
    end_time = time.time()
    print(f"fine-tuning run-{k}:",end_time - start_time,"seconds")

2026-01-06 10:00:19,447	WARNING algorithm_config.py:596 -- Cannot create MERLIONConfig from given `config_dict`! Property __stdout_file__ not supported.
2026-01-06 10:00:19,447	WARNING algorithm_config.py:596 -- Cannot create MERLIONConfig from given `config_dict`! Property __stdout_file__ not supported.
2026-01-06 10:00:19,487	INFO algorithm.py:506 -- Current log_level is WARN. For more information, set 'log_level': 'INFO' / 'DEBUG' or use the -v and -vv flags.
2026-01-06 10:00:19,487	INFO algorithm.py:506 -- Current log_level is WARN. For more information, set 'log_level': 'INFO' / 'DEBUG' or use the -v and -vv flags.



=== Meta-policy 0 ===


(RolloutWorker pid=2878) 2026-01-06 10:00:22,767	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=2878) 2026-01-06 10:00:22,767	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=2878) 2026-01-06 10:00:22,767	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(RolloutWorker pid=2878) 2026-01-06 10:00:22,767	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
2026-0

[skip+load] i000_m-1: /home/rifnyrachman7/_merlion/training_data/moderate_sc/finetuning_sb3-8/i000_m-1 already exists, loading existing results into summaries.
[skip+load] i000_m00: /home/rifnyrachman7/_merlion/training_data/moderate_sc/finetuning_sb3-8/i000_m00 already exists, loading existing results into summaries.
[skip+load] i000_m01: /home/rifnyrachman7/_merlion/training_data/moderate_sc/finetuning_sb3-8/i000_m01 already exists, loading existing results into summaries.

=== Meta-policy 1 ===


(RolloutWorker pid=3291) 2026-01-06 10:00:34,045	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=3291) 2026-01-06 10:00:34,045	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=3291) 2026-01-06 10:00:34,046	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(RolloutWorker pid=3291) 2026-01-06 10:00:34,046	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(Rollo

...

/home/rifnyrachman7/_metamorl/algo_ray/rllib/algorithms/maml/evaluate_merlion.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


[warn] No overlapping keys between RLlib policy and SB3 policy; using SB3 init.
[warn] No overlapping keys between RLlib policy and SB3 policy; using SB3 init.

=== Meta-policy 17 ===


(RolloutWorker pid=25919) 2026-01-06 21:45:16,546	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=25919) 2026-01-06 21:45:16,546	WARNING env.py:156 -- Your env doesn't have a .spec.max_episode_steps attribute. Your horizon will default to infinity, and your environment will not be reset.
(RolloutWorker pid=25919) 2026-01-06 21:45:16,546	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(RolloutWorker pid=25919) 2026-01-06 21:45:16,546	WARNING env.py:166 -- Your env reset() method appears to take 'seed' or 'return_info' arguments. Note that these are not yet supported in RLlib. Seeding will take place using 'env.seed()' and the info dict will not be returned from reset.
(R

[warn] No overlapping keys between RLlib policy and SB3 policy; using SB3 init.


/home/rifnyrachman7/_metamorl/algo_ray/rllib/algorithms/maml/evaluate_merlion.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


### Load the model to print policy

In [23]:
ENV_ID     = "testSC"
run = 9
MODEL_PATH = f"/home/rifnyrachman7/_merlion/training_data/moderate_sc/finetuning_sb3-{run}/i003_m01/sb3_model.zip"  # <- your file

env = gym.make(ENV_ID, max_steps=100)
model = PPO.load(MODEL_PATH, env=env)

# Try to obtain human-readable action labels, otherwise use a[i]
action_labels = None
ue = getattr(env, "unwrapped", env)
for attr in ("get_action_names", "action_names", "action_labels"):
    if hasattr(ue, attr):
        names = getattr(ue, attr)
        action_labels = names() if callable(names) else names
        break

CSV_OUT    = f"/home/rifnyrachman7/_merlion/visualisation/action_mod_merlion-{run}.csv"
obs, info = env.reset()
rows = []
t = 0
while True:
    action, _ = model.predict(obs, deterministic=False)
    action = np.array(action).ravel()

    row = {"t": t}
    if action_labels and len(action_labels) == action.size:
        for name, val in zip(action_labels, action):
            row[name] = float(val)
    else:
        for i, val in enumerate(action):
            row[f"a{i}"] = float(val)

    # Optional: capture reward vector if the env provides it
    mo = info.get("mo_reward")
    if mo is not None and len(mo) >= 3:
        row["profit"]    = float(mo[0])
        row["emission"]  = float(mo[1])
        row["inequality"]= float(mo[2])

    rows.append(row)

    obs, reward, terminated, truncated, info = env.step(action)
    if terminated or truncated:
        break
    t += 1

env.close()

df = pd.DataFrame(rows)
df.to_csv(CSV_OUT, index=False)
print(f"Saved action plan to {CSV_OUT}")

Saved action plan to /home/rifnyrachman7/_merlion/visualisation/action_mod_merlion-9.csv
